# Latency Execution Engine - Data Download

**Run this notebook in Google Colab to download all data.**

Data Strategy:

- **Klines (1-min OHLCV):** BTCUSDT, ETHUSDT, SOLUSDT - 2020-2024 (5 years, ~2GB)
- **Tick-level trades:** BTCUSDT - 2023-2024 (~10-15GB)

After download, data is saved to Google Drive for persistent storage.


In [ ]:
# ============================================
# CELL 1: Mount Google Drive for persistent storage
# ============================================
from google.colab import drive
drive.mount('/content/drive')

# Create project directory in Drive
import os
PROJECT_DIR = '/content/drive/MyDrive/latency-execution-engine'
RAW_DIR = f'{PROJECT_DIR}/data/raw'
os.makedirs(f'{RAW_DIR}/klines', exist_ok=True)
os.makedirs(f'{RAW_DIR}/trades', exist_ok=True)
print(f'Project directory: {PROJECT_DIR}')
print(f'Raw data directory: {RAW_DIR}')

In [ ]:
# ============================================
# CELL 2: Download engine (run this cell once)
# ============================================
import io
import time
import zipfile
import requests
from pathlib import Path
from tqdm.notebook import tqdm

BASE_URL = 'https://data.binance.vision/data/spot/monthly'


def generate_months(start: str, end: str) -> list[tuple[int, int]]:
    """Generate (year, month) tuples from 'YYYY-MM' to 'YYYY-MM'."""
    sy, sm = map(int, start.split('-'))
    ey, em = map(int, end.split('-'))
    months = []
    y, m = sy, sm
    while (y, m) <= (ey, em):
        months.append((y, m))
        m += 1
        if m > 12:
            m = 1
            y += 1
    return months


def download_file(url: str, output_path: Path, retries: int = 3) -> bool:
    """Download a zip, extract CSV. Returns True on success."""
    if output_path.exists():
        return True  # Already downloaded (checkpoint)

    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(url, timeout=120)
            if resp.status_code == 404:
                return False
            if resp.status_code == 451:
                print(f'  Region-restricted. Try VPN.')
                return False
            resp.raise_for_status()

            with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
                csv_files = [f for f in zf.namelist() if f.endswith('.csv')]
                if not csv_files:
                    return False
                # Extract to parent dir, then rename
                output_path.parent.mkdir(parents=True, exist_ok=True)
                data = zf.read(csv_files[0])
                output_path.write_bytes(data)
            return True

        except Exception as e:
            if attempt < retries:
                time.sleep(2 ** attempt)
            else:
                print(f'  Failed: {url.split("/")[-1]} - {e}')
                return False
    return False


def download_klines(symbol: str, start: str, end: str,
                    interval: str = '1m', output_base: str = RAW_DIR):
    """Download all kline months for a symbol."""
    months = generate_months(start, end)
    out_dir = Path(output_base) / 'klines' / symbol
    out_dir.mkdir(parents=True, exist_ok=True)

    success = 0
    for y, m in tqdm(months, desc=f'{symbol} klines'):
        fname = f'{symbol}-{interval}-{y}-{m:02d}.csv'
        url = f'{BASE_URL}/klines/{symbol}/{interval}/{symbol}-{interval}-{y}-{m:02d}.zip'
        if download_file(url, out_dir / fname):
            success += 1
        time.sleep(0.2)

    print(f'{symbol} klines: {success}/{len(months)} months')
    return success


def download_trades(symbol: str, start: str, end: str,
                    output_base: str = RAW_DIR):
    """Download all trade months for a symbol. WARNING: ~500MB/month!"""
    months = generate_months(start, end)
    out_dir = Path(output_base) / 'trades' / symbol
    out_dir.mkdir(parents=True, exist_ok=True)

    success = 0
    for y, m in tqdm(months, desc=f'{symbol} trades'):
        fname = f'{symbol}-trades-{y}-{m:02d}.csv'
        url = f'{BASE_URL}/trades/{symbol}/{symbol}-trades-{y}-{m:02d}.zip'
        if download_file(url, out_dir / fname):
            success += 1
        time.sleep(0.3)

    print(f'{symbol} trades: {success}/{len(months)} months')
    return success


print('Download engine ready!')

In [ ]:
# ============================================
# CELL 3: Download KLINES - All 3 symbols, 2020-2024
# ============================================
# Estimated: ~2 GB total, ~15-30 minutes
#
# This downloads 1-minute OHLCV bars:
#   BTCUSDT: 60 months (2020-01 to 2024-12)
#   ETHUSDT: 60 months
#   SOLUSDT: 60 months (available from ~2020-09)

KLINE_SYMBOLS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']
KLINE_START = '2020-01'
KLINE_END = '2024-12'

print('DOWNLOADING KLINES (1-minute OHLCV)')
print(f'   Symbols: {KLINE_SYMBOLS}')
print(f'   Range:   {KLINE_START} → {KLINE_END}')
print(f'   Output:  {RAW_DIR}/klines/')
print()

for symbol in KLINE_SYMBOLS:
    download_klines(symbol, KLINE_START, KLINE_END)
    print()

In [ ]:
# ============================================
# CELL 4: Download TRADES - BTCUSDT only, 2023-2024
# ============================================
#  WARNING: This is ~10-15 GB. Takes 1-2 hours.
#     Make sure you have enough Google Drive space.
#     You can skip this cell initially and come back later.
#
# Trade data = every individual executed transaction
# Fields: trade_id, price, quantity, quote_qty, timestamp, is_buyer_maker
#
# This is what makes the project TIER 1:
#   - Enables realistic market impact modeling
#   - Can compute actual bid-ask spread from trade direction
#   - Can calibrate Almgren-Chriss impact parameters from real data

TRADE_SYMBOL = 'BTCUSDT'
TRADE_START = '2023-01'
TRADE_END = '2024-12'

print('DOWNLOADING TICK-LEVEL TRADES')
print(f'   Symbol:  {TRADE_SYMBOL}')
print(f'   Range:   {TRADE_START} → {TRADE_END}')
print(f'   Output:  {RAW_DIR}/trades/')
print(f'    Expected size: ~10-15 GB')
print()

download_trades(TRADE_SYMBOL, TRADE_START, TRADE_END)

In [ ]:
# ============================================
# CELL 5: Verify downloads & print summary
# ============================================
from pathlib import Path

def print_summary(base_dir: str):
    base = Path(base_dir)
    total_size = 0
    total_files = 0

    print(f"\n{'='*60}")
    print(f"  DOWNLOAD SUMMARY")
    print(f"{'='*60}")

    for data_type in ['klines', 'trades']:
        type_dir = base / data_type
        if not type_dir.exists():
            continue
        for symbol_dir in sorted(type_dir.iterdir()):
            if not symbol_dir.is_dir():
                continue
            files = list(symbol_dir.glob('*.csv'))
            size = sum(f.stat().st_size for f in files)
            total_size += size
            total_files += len(files)
            print(f"  {data_type:8s} | {symbol_dir.name:10s} | "
                  f"{len(files):3d} files | {size/1024/1024:,.1f} MB")

    print(f"{'='*60}")
    print(f"  TOTAL: {total_files} files, {total_size/1024/1024/1024:.2f} GB")
    print(f"{'='*60}")

print_summary(RAW_DIR)

In [ ]:
# ============================================
# CELL 6: Quick sanity check - load and inspect one file
# ============================================
import pandas as pd
import numpy as np

# Kline columns (Binance CSVs have no header)
KLINE_COLS = [
    'open_time', 'open', 'high', 'low', 'close', 'volume',
    'close_time', 'quote_volume', 'num_trades',
    'taker_buy_vol', 'taker_buy_quote', 'ignore'
]

# Load one month of BTCUSDT klines
sample_file = Path(RAW_DIR) / 'klines' / 'BTCUSDT' / 'BTCUSDT-1m-2023-01.csv'

if sample_file.exists():
    df = pd.read_csv(sample_file, header=None, names=KLINE_COLS)
    df['timestamp'] = pd.to_datetime(df['open_time'], unit='ms', utc=True)

    print(f'Rows:        {len(df):,}')
    print(f'Date range:  {df["timestamp"].min()} → {df["timestamp"].max()}')
    print(f'Price range: ${df["close"].min():,.2f} → ${df["close"].max():,.2f}')
    print(f'Avg volume:  {df["volume"].mean():,.2f} BTC/min')
    print(f'Avg trades:  {df["num_trades"].mean():,.0f} trades/min')
    print(f'\nFirst 3 rows:')
    display(df[['timestamp', 'open', 'high', 'low', 'close', 'volume', 'num_trades']].head(3))
else:
    print(f'File not found: {sample_file}')
    print('Run Cell 3 first to download klines.')

## What's Next?

After downloading, copy the `data/raw/` folder structure to your local project:

```
latency-execution-engine/
  data/
    raw/
      klines/
        BTCUSDT/   ← ~60 CSV files
        ETHUSDT/   ← ~60 CSV files
        SOLUSDT/   ← ~55 CSV files
      trades/
        BTCUSDT/   ← ~24 CSV files (HUGE)
```

Or keep it in Drive and mount Drive when running Colab notebooks for analysis.

**Remember:** Data files are in `.gitignore` - NEVER push them to GitHub.
